# USCIS Analysis Database Schema

This project loads manually collected USCIS files into the local PostgreSQL database `uscis_analysis`.

The database is split into four schemas:

- `uscis_raw`: source file registry and raw workbook/PDF/CSV extraction trace.
- `uscis_dim`: lookup/dimension tables for forms, countries, statuses, report families, and I-140 preference categories.
- `uscis_fact`: normalized numerical facts extracted from the source files.
- `uscis_mart`: analysis-ready views.

## Current Loaded State

Last verified local load:

| Object | Row count |
|---|---:|
| `uscis_raw.source_files` | 79 |
| `uscis_raw.workbook_sheets` | 155 |
| `uscis_raw.sheet_cells` | 119,329 |
| `uscis_dim.forms` | 143 |
| `uscis_dim.countries` | 17 |
| `uscis_fact.i140_status_counts` | 30,434 |
| `uscis_fact.form_status_counts` | 7,589 |

Local raw file folder:

`C:\Users\kelav\OneDrive\Документы\projects\uscis_analysis\data\raw\raw_by_hands`

Detected local source files in that folder:

| Extension | Count |
|---|---:|
| `.csv` | 54 |
| `.xlsx` | 24 |
| `.pdf` | 45 |

Not every file is loaded as a separate source. If a PDF appears to duplicate a CSV/XLSX with the same normalized name, the parser prefers the structured CSV/XLSX and skips the PDF. Exact duplicate file hashes are also skipped.

## Raw Schema

### `uscis_raw.source_files`

One row per loaded source file.

Important columns:

- `id`: source file primary key.
- `report_family`: logical report type, such as `all_forms`, `fy_quarter_status`, `preference_country`, or `country_category`.
- `file_name`: original local file name.
- `source_url`: local loads use a `local_hand://...` pseudo-URL.
- `local_path`: absolute local path to the parsed file.
- `report_fiscal_year`, `report_quarter`: fiscal year/quarter inferred from the file name.
- `snapshot_date`: current observation date used for analysis. For local files this is currently inferred as the fiscal quarter end date, unless `SNAPSHOT_DATE` is explicitly supplied.
- `file_hash_sha256`: unique content hash.
- `parser_version`, `parse_status`, `parse_notes`: parser metadata.

### `uscis_raw.workbook_sheets`

One row per worksheet, CSV pseudo-sheet, or PDF text pseudo-sheet.

Important columns:

- `source_file_id`: FK to `uscis_raw.source_files`.
- `sheet_name`: worksheet name, `CSV`, or `PDF Text`.
- `sheet_index`: zero-based sheet index.
- `max_row`, `max_column`: extracted shape.
- `detected_table_type`: usually same logical family as `source_files.report_family`.
- `detection_confidence`: coarse parser confidence.

### `uscis_raw.sheet_cells`

Raw extraction audit table.

For XLSX/CSV this stores cell-like values. For PDFs this stores extracted text lines as one-column pseudo-cells.

Important columns:

- `sheet_id`: FK to `uscis_raw.workbook_sheets`.
- `row_num`, `col_num`, `cell_address`: source location.
- `raw_value`, `normalized_value`: original and normalized text.
- `is_merged`, `merged_range`: XLSX merged-cell trace where available.

### `uscis_raw.extraction_corrections`

Reserved table for future manual corrections. It is not the main fact store.

Use this when a parsed value is corrected manually and the reason needs to be recorded.

## Dimension Schema

### `uscis_dim.forms`

Form lookup table.

Examples:

- `I-140`
- `I-130`
- `I-485`
- `TOTAL`

`form_status_counts` can contain many forms. `i140_status_counts` is specifically for I-140 and joins to the `I-140` form row.

### `uscis_dim.preference_categories`

I-140 employment preference/category lookup.

Core categories:

- `TOTAL`
- `EB1`, `E11`, `E12`, `E13`
- `EB2`, `E21`, `NIW`
- `EB3`, `E31`, `E32`, `EW3`
- `OTHER_UNKNOWN`

`parent_category_code` links subcategories to broader EB preference groups.

### `uscis_dim.case_statuses`

Status lookup.

Current statuses:

- `received`
- `approved`
- `denied`
- `pending`
- `pending_other`

Note: USCIS wording differs across reports. `pending_other` is used for labels such as `Pending, Other`.

### `uscis_dim.countries`

Country/aggregate lookup.

`ALL` means All Countries. Other country codes are normalized from USCIS labels and are not guaranteed to be ISO codes.

### `uscis_dim.report_families`

Logical report family lookup.

Current families:

- `all_forms`: service-wide quarterly form counts by form number.
- `fy_quarter_status`: I-140 by fiscal year, quarter, and case status.
- `preference_country`: I-140 receipts/current status by preference and country.
- `country_category`: I-140 receipts or approvals by country and EB category.

## Fact Schema

### `uscis_fact.i140_status_counts`

Main normalized I-140 fact table.

This table stores I-140 facts by:

- source file
- sheet
- EB category
- country
- status
- cohort fiscal year
- cohort quarter, if present
- report fiscal year/quarter
- snapshot date

Important columns:

- `source_file_id`: FK to the source file.
- `sheet_id`: FK to source sheet or pseudo-sheet.
- `form_id`: always I-140 for this fact table.
- `category_id`: EB/I-140 category dimension.
- `country_id`: country or `ALL`.
- `status_id`: received/approved/denied/pending/pending_other.
- `cohort_fiscal_year`, `cohort_quarter`: period the cases belong to.
- `report_fiscal_year`, `report_quarter`: report issue period inferred from file name.
- `snapshot_date`: observation date.
- `count_value`: extracted count.
- `raw_row_label`, `raw_column_label`, `raw_cell_address`: audit trace back to source layout.
- `extraction_method`, `extraction_confidence`, `reviewed`: extraction metadata.

Use this table for I-140 category/country/status analysis.

### `uscis_fact.form_status_counts`

Service-wide all-forms fact table.

This table stores counts for forms such as I-140, I-130, I-485, etc. It is the source for the consistent quarterly I-140 all-forms time series.

Important columns:

- `source_file_id`, `sheet_id`
- `form_id`
- `status_id`
- `report_fiscal_year`, `report_quarter`
- `period_scope`: `quarter` or `ytd`
- `period_quarter`: quarter number for `quarter` rows.
- `count_value`: numeric value. It is numeric rather than integer because all-forms source rows may contain processing-time decimals; the parser stores only count columns, but the table type is flexible.
- `form_category`, `form_description`
- raw source trace and extraction metadata.

Use this table for the broadest 2017-2025 I-140 quarterly series:

- received
- approved
- denied
- pending

## Mart Schema

### `uscis_mart.i140_rates_by_snapshot`

Analysis view over `uscis_fact.i140_status_counts`.

It aggregates rows by source, category, country, cohort period, and snapshot date, then calculates:

- `received`
- `approved`
- `denied`
- `pending`
- `approval_rate_received_basis`: `approved / received`
- `approval_rate_decided_basis`: `approved / (approved + denied)`
- `pending_share`: `pending / received`

This view does not use `form_status_counts`. It is for I-140 category/country reports, not the all-forms service-wide series.

## Relationship Map

```mermaid
erDiagram
    "uscis_raw.source_files" ||--o{ "uscis_raw.workbook_sheets" : has
    "uscis_raw.workbook_sheets" ||--o{ "uscis_raw.sheet_cells" : contains
    "uscis_raw.source_files" ||--o{ "uscis_fact.i140_status_counts" : produces
    "uscis_raw.workbook_sheets" ||--o{ "uscis_fact.i140_status_counts" : locates
    "uscis_dim.forms" ||--o{ "uscis_fact.i140_status_counts" : classifies
    "uscis_dim.preference_categories" ||--o{ "uscis_fact.i140_status_counts" : classifies
    "uscis_dim.countries" ||--o{ "uscis_fact.i140_status_counts" : classifies
    "uscis_dim.case_statuses" ||--o{ "uscis_fact.i140_status_counts" : classifies
    "uscis_dim.report_families" ||--o{ "uscis_fact.i140_status_counts" : classifies
    "uscis_raw.source_files" ||--o{ "uscis_fact.form_status_counts" : produces
    "uscis_raw.workbook_sheets" ||--o{ "uscis_fact.form_status_counts" : locates
    "uscis_dim.forms" ||--o{ "uscis_fact.form_status_counts" : classifies
    "uscis_dim.case_statuses" ||--o{ "uscis_fact.form_status_counts" : classifies
```

## Typical Queries

I-140 quarterly all-forms time series:

```sql
SELECT
    sf.report_fiscal_year AS fy,
    f.period_quarter AS q,
    cs.status_code,
    f.count_value
FROM uscis_fact.form_status_counts f
JOIN uscis_raw.source_files sf ON sf.id = f.source_file_id
JOIN uscis_dim.forms frm ON frm.id = f.form_id
JOIN uscis_dim.case_statuses cs ON cs.id = f.status_id
WHERE sf.report_family = 'all_forms'
  AND frm.form_code = 'I-140'
  AND f.period_scope = 'quarter'
ORDER BY fy, q, cs.status_code;
```

I-140 NIW by country from I-140 fact reports:

```sql
SELECT
    sf.file_name,
    c.country_name,
    cs.status_code,
    f.cohort_fiscal_year,
    f.cohort_quarter,
    f.count_value
FROM uscis_fact.i140_status_counts f
JOIN uscis_raw.source_files sf ON sf.id = f.source_file_id
JOIN uscis_dim.preference_categories pc ON pc.id = f.category_id
JOIN uscis_dim.countries c ON c.id = f.country_id
JOIN uscis_dim.case_statuses cs ON cs.id = f.status_id
WHERE pc.category_code = 'NIW'
ORDER BY sf.report_fiscal_year, sf.report_quarter, c.country_name, cs.status_code;
```

Snapshot rate view:

```sql
SELECT *
FROM uscis_mart.i140_rates_by_snapshot
WHERE category_code = 'NIW'
ORDER BY cohort_fiscal_year, cohort_quarter, snapshot_date;
```


In [2]:
import os
import sys
import subprocess
import importlib.util
from getpass import getpass

def ensure_package(import_name, pip_name=None):
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name or import_name])

ensure_package("psycopg2", "psycopg2-binary")

import pandas as pd
import psycopg2

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 80)
pd.set_option("display.width", 160)


In [3]:
DB_HOST = "localhost"
DB_PORT = 5432
DB_NAME = "uscis_analysis"
DB_USER = "postgres"

# ?????? ?? ????????? ? ????????: ????? ?????? ?????? ??? ??????? ?????? env PGPASSWORD.
DB_PASSWORD = os.getenv("PGPASSWORD") or getpass("PostgreSQL password: ")

conn = psycopg2.connect(
    host=DB_HOST,
    port=DB_PORT,
    dbname=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD,
)

# DataFrame names match the PostgreSQL table/view names where possible.
TABLES = {
    "source_files": "SELECT * FROM uscis_raw.source_files",
    "workbook_sheets": "SELECT * FROM uscis_raw.workbook_sheets",
    "sheet_cells": "SELECT * FROM uscis_raw.sheet_cells",
    "forms": "SELECT * FROM uscis_dim.forms",
    "preference_categories": "SELECT * FROM uscis_dim.preference_categories",
    "case_statuses": "SELECT * FROM uscis_dim.case_statuses",
    "countries": "SELECT * FROM uscis_dim.countries",
    "report_families": "SELECT * FROM uscis_dim.report_families",
    "i140_status_counts": "SELECT * FROM uscis_fact.i140_status_counts",
    "form_status_counts": "SELECT * FROM uscis_fact.form_status_counts",
    "i140_rates_by_snapshot": "SELECT * FROM uscis_mart.i140_rates_by_snapshot",
}

for df_name, sql in TABLES.items():
    globals()[df_name] = pd.read_sql_query(sql, conn)

loaded_shapes = pd.DataFrame(
    [(name, len(globals()[name]), len(globals()[name].columns)) for name in TABLES],
    columns=["dataframe", "rows", "columns"],
)
loaded_shapes


PostgreSQL password:  ········


C:\Users\kelav\AppData\Local\Temp\ipykernel_23040\3714101849.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  globals()[df_name] = pd.read_sql_query(sql, conn)
C:\Users\kelav\AppData\Local\Temp\ipykernel_23040\3714101849.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  globals()[df_name] = pd.read_sql_query(sql, conn)


,dataframe,rows,columns
0,source_files,79,19
1,workbook_sheets,155,9
2,sheet_cells,119329,10
3,forms,143,3
4,preference_categories,13,8
5,case_statuses,5,6
6,countries,17,7
7,report_families,5,4
8,i140_status_counts,30434,24
9,form_status_counts,7589,20


## Этап 1. Проверка и первичное знакомство с данными

Цель этапа — понять, что реально лежит в БД, где покрытие полное, а где есть ограничения.

Задачи:

- Посчитать количество строк в основных таблицах: `source_files`, `i140_status_counts`, `form_status_counts`, `forms`, `countries`, `case_statuses`, `preference_categories`.
- Проверить диапазон данных: минимальный и максимальный `report_fiscal_year`, доступные `report_quarter`, доступные `snapshot_date`.
- Посчитать количество source files по каждому `report_family`: `all_forms`, `fy_quarter_status`, `preference_country`, `country_category`.
- Проверить, какие годы и кварталы покрыты полностью, особенно Q1-Q4 для FY2017-FY2025.
- Проверить список стран в `uscis_dim.countries` и количество фактов по каждой стране.
- Проверить, какие категории I-140 реально есть в фактах: EB1, E11, E12, E13, EB2, E21, NIW, EB3, E31, E32, EW3, OTHER_UNKNOWN.
- Найти дубликаты на уровне логического ключа: source file, category, country, status, cohort fiscal year, cohort quarter, snapshot date.
- Проверить пропуски в ключевых колонках: `category_id`, `country_id`, `status_id`, `cohort_fiscal_year`, `snapshot_date`, `count_value`.
- Проверить отрицательные, нулевые и подозрительно большие значения `count_value`.
- Найти source files, которые дали больше всего строк фактов, и понять, являются ли они страновыми/категорийными отчетами.

In [4]:
source_files.count()
source_files.head(5)
i140_status_counts.count()

i140_status_counts.head(5)
countries.head(5)

,id,country_name,iso_alpha2,iso_alpha3,uscis_country_label,notes,country_code
0,126,Chinese Nationals,None,None,Chinese Nationals,None,CHINESE_NATIONALS
1,3,India,None,None,India,None,INDIA
2,28,Brazil_FY24,None,None,Brazil_FY24,None,BRAZIL_FY24
3,4,China,None,None,China,None,CHINA
4,30,Philippines_FY24,None,None,Philippines_FY24,None,PHILIPPINES_FY24


## Этап 2. Простая квартальная динамика I-140 по all-forms

Для этого этапа основной источник — `uscis_fact.form_status_counts`, потому что он дает наиболее стабильную квартальную серию для I-140 в целом.

Задачи:

- Построить квартальную динамику `received` по I-140 за FY2017-FY2025.
- Построить квартальную динамику `approved`, `denied`, `pending` по I-140.
- Построить line chart: `received`, `approved`, `denied` по кварталам.
- Построить отдельный график `pending`, потому что масштаб backlog может быть другим.
- Посчитать годовые totals по I-140: `received`, `approved`, `denied`, `pending`.
- Построить годовой bar chart по `received`.
- Построить stacked bar chart по годам: `approved`, `denied`, `pending`.
- Найти квартал с максимальным количеством `received`.
- Найти квартал с максимальным количеством `denied`.
- Найти квартал с максимальным `pending backlog`.
- Посчитать quarter-over-quarter change для `received`: абсолютное и процентное изменение.
- Посчитать year-over-year change для `received` по одинаковым кварталам.
- Проверить сезонность: среднее количество `received` по Q1, Q2, Q3, Q4.
- Построить heatmap fiscal year x quarter для `received`.
- Построить heatmap fiscal year x quarter для `denied`.
- Построить heatmap fiscal year x quarter для `pending`.


Источник: uscis_fact.form_status_counts, только:

form_code = 'I-140'
period_scope = 'quarter'
статусы received, approved, denied, pending
*СТРОИМ*
- динамику received по кварталам;
- approved / denied / pending;
- approval rate;
- denial rate;
- pending share;
- YoY / QoQ изменения;
- heatmap year x quarter.

Колонки: fiscal_year
fiscal_quarter
received
approved
denied
pending
approval_rate_received_basis
denial_rate_received_basis
approval_rate_decided_basis
denial_rate_decided_basis
pending_share


In [5]:

# Собираем рабочий df из трех таблиц.
fsc = form_status_counts.copy()

fsc = fsc.merge(
    forms[['id', 'form_code', 'form_name']],
    left_on='form_id',
    right_on='id',
    how='left',
    suffixes=('', '_form')
)

fsc = fsc.merge(
    case_statuses[['id', 'status_code', 'status_name']],
    left_on='status_id',
    right_on='id',
    how='left',
    suffixes=('', '_status')
)

fsc = fsc.merge(
    source_files[['id', 'report_family', 'file_name']],
    left_on='source_file_id',
    right_on='id',
    how='left',
    suffixes=('', '_source')
)

In [6]:
fsc.head(5)
fsc[['report_fiscal_year', 'report_quarter', 'period_scope', 'period_quarter',
     'form_code', 'status_code', 'count_value', 'form_description']]
i140_q = fsc[
    (fsc['form_code'] == 'I-140') &
    (fsc['period_scope'] == 'quarter')
].copy()

In [7]:
i140_q
i140_q.groupby('status_code')['count_value'].count()
i140_pivot = i140_q.pivot_table(
    index=['report_fiscal_year', 'period_quarter'],
    columns='status_code',
    values='count_value',
    aggfunc='sum'
).reset_index()
i140_pivot

status_code,report_fiscal_year,period_quarter,approved,denied,pending,received
0,2017,1.0,30852.000,2032.000,56112.0,38209.0
1,2017,2.0,30852.000,2032.000,56112.0,38209.0
2,2017,3.0,30852.000,2032.000,56112.0,38209.0
3,2017,4.0,30852.000,2032.000,56105.0,38209.0
4,2018,1.0,27753.000,2488.000,58776.0,27581.0
5,2018,2.0,27753.000,2488.000,56701.0,27581.0
6,2018,3.0,27753.000,2488.000,56701.0,27581.0
7,2018,4.0,27753.000,2488.000,56700.0,27581.0
8,2019,1.0,119997.000,9354.000,143463.0,116079.0
9,2019,2.0,110917.000,10034.000,125981.0,95929.0


In [10]:
i_140_rates = i140_pivot.copy()
i_140_rates['approval_rate_%'] = i_140_rates.approved/i_140_rates.received*100
i_140_rates['denial_rate_%'] = i_140_rates.denied/i_140_rates.received*100
i_140_rates['pending_rate_%'] = i_140_rates.pending/i_140_rates.received*100

i_140_rates


status_code,report_fiscal_year,period_quarter,approved,denied,pending,received,approval_rate_%,denial_rate_%,pending_rate_%
0,2017,1.0,30852.000,2032.000,56112.0,38209.0,80.745374,5.318119,146.855453
1,2017,2.0,30852.000,2032.000,56112.0,38209.0,80.745374,5.318119,146.855453
2,2017,3.0,30852.000,2032.000,56112.0,38209.0,80.745374,5.318119,146.855453
3,2017,4.0,30852.000,2032.000,56105.0,38209.0,80.745374,5.318119,146.837133
4,2018,1.0,27753.000,2488.000,58776.0,27581.0,100.623618,9.020703,213.103223
5,2018,2.0,27753.000,2488.000,56701.0,27581.0,100.623618,9.020703,205.579928
6,2018,3.0,27753.000,2488.000,56701.0,27581.0,100.623618,9.020703,205.579928
7,2018,4.0,27753.000,2488.000,56700.0,27581.0,100.623618,9.020703,205.576303
8,2019,1.0,119997.000,9354.000,143463.0,116079.0,103.375288,8.058305,123.590830
9,2019,2.0,110917.000,10034.000,125981.0,95929.0,115.624055,10.459819,131.327336


## Этап 3. Approval, denial и pending rates

Цель этапа — увидеть разные способы расчета rates и не смешивать “received basis” с “decided basis”.

Задачи:

- Посчитать `approved / received` по кварталам для I-140.
- Посчитать `denied / received` по кварталам для I-140.
- Посчитать `pending / received` по кварталам для I-140.
- Посчитать `approved / (approved + denied)` по кварталам.
- Сравнить `approved / received` и `approved / (approved + denied)`.
- Построить line chart approval rate на received basis.
- Построить line chart approval rate на decided basis.
- Построить line chart pending share.
- Найти кварталы, где pending share превышает 50%, 70%, 90%.
- Найти кварталы, где denial rate на decided basis был максимальным.
- Сравнить FY2024 и FY2025 по `received`, `approved`, `denied`, `pending`, approval rate, denial rate, pending share.
- Построить rolling average для `received` на 4 квартала.
- Построить rolling average для denial rate на 4 квартала.
- Проверить, не ломают ли свежие кварталы общую картину из-за pending, отдельно сравнив FY2017-FY2022 и FY2023-FY2025.